# Top 5000 Spotify Tracks Downloader

Loads batch results and stream results, merges them with the original dataset (to get popularity), deduplicates columns, sorts by popularity, and downloads previews and cover arts.

In [ ]:
import polars as pl
from pathlib import Path
import urllib.request
from urllib.parse import urlparse
import re
from tqdm import tqdm
import requests
import os
import time
import random
from concurrent.futures import ThreadPoolExecutor, as_completed
import lyricsgenius
import spotify_scraper as sps


In [ ]:
# --- Config ---
BASE_DIR = Path('../spotify_v2')
BATCH_RESULTS_DIR = BASE_DIR / 'batch_results'
STREAMS_DIR = BASE_DIR / 'batch_results_streams'
ORIGINAL_DATA = BASE_DIR / 'spotify_data.parquet'

OUTPUT_DIR = Path('.')
DATA_DIR = OUTPUT_DIR / 'data'
COVERS_DIR = DATA_DIR / 'covers'
PREVIEWS_DIR = DATA_DIR / 'previews'

COVERS_DIR.mkdir(parents=True, exist_ok=True)
PREVIEWS_DIR.mkdir(parents=True, exist_ok=True)

# --- Load Data ---
print("Loading batch files...")
batch_files = list(BATCH_RESULTS_DIR.glob('*.parquet'))
df_batches = pl.concat([pl.read_parquet(f) for f in batch_files], how='diagonal_relaxed')
print(f"Batches shape: {df_batches.shape}")

print("Loading stream files...")
stream_files = list(STREAMS_DIR.glob('*.parquet'))
df_streams = pl.concat([pl.read_parquet(f) for f in stream_files], how='diagonal_relaxed')
print(f"Streams shape: {df_streams.shape}")

print("Loading original dataset for popularity...")
df_data = pl.read_parquet(ORIGINAL_DATA)
print(f"Original data shape: {df_data.shape}")

In [ ]:
# --- Merge Data ---
# Merge batches and streams
df_merged = df_batches.join(df_streams, on='track_id', how='left')

# Merge with original data to get popularity
df_merged = df_merged.join(df_data, on='track_id', how='left')


# Keep only rows where popularity is available
df_merged = df_merged.filter(pl.col('popularity').is_not_null())

# --- Sort by Popularity and Get Top 5000 ---
# We sort by popularity descending.
# If you also want to take stream counts into account as a secondary measure: sort(['popularity', 'stream_count'], descending=[True, True])
df_top5000 = df_merged.sort('popularity', descending=True).head(5000)
print(f"Top 5000 shape: {df_top5000.shape}")

# --- Output Files ---
parquet_out = 'spotify_5000.parquet'
df_top5000.write_parquet(parquet_out)
print(f"Saved {parquet_out}")

In [ ]:
# --- Downloader Functions (based on spotify_scraper_v1.ipynb) ---
def extract_cover_url(album_value, resolution='300'):
    if album_value is None:
        return None
    if isinstance(album_value, str):
        matches = re.findall(r"https?://[^'\"\\s,]+", album_value)
    elif isinstance(album_value, dict) and 'images' in album_value:
        matches = [img['url'] for img in album_value['images'] if 'url' in img]
    else:
        matches = re.findall(r"https?://[^'\"\\s,]+", str(album_value))

    if not matches:
        return None
    # Try to select the URL with the requested resolution
    for url in matches:
        if f'/{resolution}x{resolution}/' in url:
            return url
    # Fallback to last match
    return matches[-1]

def infer_image_extension(url):
    suffix = Path(urlparse(url).path).suffix.lower()
    if suffix in {'.jpg', '.jpeg', '.png', '.webp'}:
        return suffix
    return '.jpg'

def download_binary(url, output_path, timeout=30):
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            output_path.write_bytes(resp.read())
        return True
    except Exception:
        return False

In [ ]:
# --- Download Covers ---
COVER_RESOLUTION = '300'  # You can change to '640' or '64'
cover_status = {}

for row in tqdm(df_top5000.to_dicts(), desc='Covers'):
    track_id = row['track_id']
    cover_url = extract_cover_url(row.get('album'), COVER_RESOLUTION)
    if not cover_url:
        cover_status[track_id] = False
        continue
    
    ext = infer_image_extension(cover_url)
    output_path = COVERS_DIR / f'{track_id}{ext}'
    
    if output_path.exists():
        cover_status[track_id] = True
        continue
    
    cover_status[track_id] = download_binary(cover_url, output_path)

In [ ]:
# --- Download Previews ---
preview_status = {}

for row in tqdm(df_top5000.to_dicts(), desc='Previews'):
    track_id = row['track_id']
    preview_url = row.get('preview_url')
    if not preview_url or str(preview_url).strip() == '':
        preview_status[track_id] = False
        continue
    
    output_path = PREVIEWS_DIR / f'{track_id}.mp3'
    if output_path.exists():
        preview_status[track_id] = True
        continue
    
    preview_status[track_id] = download_binary(preview_url, output_path)

In [ ]:
# --- Generate CSV Report ---
report_df = (
    df_top5000.select(['track_id'])
    .unique(subset=['track_id'])
    .with_columns([
        pl.col('track_id').map_elements(lambda tid: cover_status.get(tid, False), return_dtype=pl.Boolean).alias('cover_downloaded'),
        pl.col('track_id').map_elements(lambda tid: preview_status.get(tid, False), return_dtype=pl.Boolean).alias('preview_downloaded'),
    ])
)

report_path = OUTPUT_DIR / 'download_report.csv'
report_df.write_csv(report_path)
print(f"Download report saved: {report_path.resolve()}")
print(report_df.head())

In [15]:
# # --- Add REAL Album Name and Lyrics Language via Scraping ---
# import spotifyscraper as sps  # ensure using correct name if needed
# from syrics.api import Spotify
# import lyricsgenius

# SP_DC = os.getenv("SPOTIFY_SP_DC")
# GENIUS_TOKEN = os.getenv("GENIUS_ACCESS_TOKEN")

# # Initialize clients
# lyrics_client = None
# if SP_DC:
#     try:
#         lyrics_client = Spotify(SP_DC)
#     except Exception as e:
#         print(f"Spotify client failed: {e}")

# genius_client = None
# if GENIUS_TOKEN:
#     try:
#         genius_client = lyricsgenius.Genius(GENIUS_TOKEN, verbose=False, timeout=5)
#     except Exception as e:
#         print(f"Genius client failed: {e}")

# def scrape_metadata(row):
#     track_id = row['track_id']
#     track_name = row.get('track_name', '')
#     artist_name = row.get('artist_name', row.get('artists', ''))
    
#     # Try to clean up artist name if it is a list or string of list
#     if isinstance(artist_name, str) and artist_name.startswith('['):
#         try:
#             import ast
#             artist_list = ast.literal_eval(artist_name)
#             if artist_list:
#                 artist_name = artist_list[0]
#         except:
#             pass
#     elif isinstance(artist_name, list) and artist_name:
#         artist_name = artist_name[0]
    
#     res = {'track_id': track_id, 'album_name': '', 'lyrics_language': '', 'lyrics': ''}
    
#     # 1. Fetch Album Name (Spotify Embed / OEmbed is often reliable for public metadata)
#     try:
#         # Fallback to LRCLib for album name as it is very fast and public
#         r = requests.get("https://lrclib.net/api/search", 
#                          params={"track_name": track_name, "artist_name": artist_name},
#                          timeout=5)
#         if r.status_code == 200:
#             data = r.json()
#             if data and isinstance(data, list):
#                 res['album_name'] = data[0].get('albumName', '')
#     except:
#         pass

#     # 2. Fetch Lyrics and Language (Genius)
#     if genius_client and (not res['album_name'] or not res['lyrics_language']):
#         try:
#             song = genius_client.search_song(track_name, artist_name)
#             if song:
#                 if not res['album_name']:
#                     res['album_name'] = getattr(song, 'album', '')
#                     if hasattr(res['album_name'], 'name'):
#                         res['album_name'] = res['album_name'].name
                
#                 # Language detection from Genius
#                 # lyricsgenius objects sometimes have language in the raw data
#                 raw = getattr(song, '_body', {})
#                 res['lyrics_language'] = raw.get('language', '')
#                 res['lyrics'] = song.lyrics
#         except:
#             pass
            
#     # 3. Last resort for album name: Spotify OEmbed
#     if not res['album_name']:
#         try:
#             r = requests.get(f"https://open.spotify.com/oembed?url=spotify:track:{track_id}", timeout=5)
#             if r.status_code == 200:
#                 # OEmbed title is often "Song Name" or "Song Name by Artist"
#                 # It doesn't always have album. 
#                 pass
#         except:
#             pass

#     return res

# print("Scraping detailed metadata (album_name, lyrics_language) for 5000 tracks...")
# metadata_results = []
# rows = df_top5000.to_dicts()

# with ThreadPoolExecutor(max_workers=10) as executor:
#     futures = {executor.submit(scrape_metadata, row): row for row in rows}
#     for future in tqdm(as_completed(futures), total=len(futures), desc='Metadata'):
#         metadata_results.append(future.result())

# # Merge results
# df_metadata = pl.from_dicts(metadata_results)
# df_top5000 = df_top5000.join(df_metadata, on='track_id', how='left')

# Save updated parquet
parquet_out = 'spotify_5000.parquet'
df_top5000.write_parquet(parquet_out)
print(f"Saved {parquet_out} with album_name and lyrics_language.")



Saved spotify_5000.parquet with album_name and lyrics_language.
